# Embedding Generation for Stack Overflow Questions

**What this does:** Convert 498K cleaned questions into 384-dimensional vectors using `all-MiniLM-L6-v2`

**Input:** `questions_cleaned.parquet` (from cleaning step)

**Output:**
- `question_embeddings.npy` — (498644, 384) numpy array
- `question_ids.npy` — ID mapping so we always know which embedding belongs to which question

**Runtime:** ~15-20 min on GPU, ~2-3 hours on CPU

---
## 1. Setup

In [1]:
!pip install sentence-transformers pyarrow -q

In [2]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import time
import os

print('Libraries imported!')

Libraries imported!


---
## 2. Load Cleaned Data

In [3]:

DATA_DIR = '../Dataset_Cleaned/questions_cleaned.parquet'

# Using parquet (faster loading, includes all columns)
df = pd.read_parquet(os.path.join(DATA_DIR))

print(f'Loaded: {len(df):,} questions')
print(f'Columns: {list(df.columns)}')
print(f'\nSample:')
print(f'  Title: {df.iloc[0]["title"][:80]}')
print(f'  Tags:  {df.iloc[0]["tags"]}')
print(f'  Body:  {df.iloc[0]["body"][:100]}...')

Loaded: 498,644 questions
Columns: ['id', 'title', 'body', 'code_blocks', 'tags', 'primary_tag', 'score', 'view_count', 'answer_count', 'has_accepted_answer', 'creation_date', 'title_length', 'body_length', 'body_word_count', 'has_code', 'num_tags']

Sample:
  Title: Why is processing a sorted array faster than processing an unsorted array?
  Tags:  java,c++,performance,cpu-architecture,branch-prediction
  Body:  Here is a piece of C++ code that shows some very peculiar behavior. For some strange reason, sorting...


---
## 3. Create Embedding Input Text

Format: `{title} [SEP] {tags} [SEP] {first 200 words of body}`

- **[SEP]** is a special separator token that BERT-family models understand
- **First 200 words** because the model's max input is 256 tokens (~200 words). Anything beyond gets truncated internally anyway, so we save tokenization time by pre-truncating.

In [4]:
def truncate_body(text, max_words=200):
    """Keep only the first max_words words of body text."""
    if pd.isna(text) or not text:
        return ''
    words = text.split()
    return ' '.join(words[:max_words])


# Build embedding input: title [SEP] tags [SEP] body (truncated)
df['embedding_text'] = (
    df['title'].fillna('') + 
    ' [SEP] ' + 
    df['tags'].fillna('') + 
    ' [SEP] ' + 
    df['body'].fillna('').apply(truncate_body)
)

# Check a sample
print('Sample embedding text:')
print('=' * 80)
print(df['embedding_text'].iloc[0][:300])
print('...')
print(f'\nTotal characters: {len(df["embedding_text"].iloc[0])}')
print(f'Total words: {len(df["embedding_text"].iloc[0].split())}')

Sample embedding text:
Why is processing a sorted array faster than processing an unsorted array? [SEP] java,c++,performance,cpu-architecture,branch-prediction [SEP] Here is a piece of C++ code that shows some very peculiar behavior. For some strange reason, sorting the data ( before the timed region) miraculously makes t
...

Total characters: 1284
Total words: 215


---
## 4. Load Model & Check Device

In [6]:
# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('No GPU detected — running on CPU.')
    print('This will take ~2-3 hours. If you have an NVIDIA GPU,')
    print('install CUDA toolkit + pytorch-cuda for much faster encoding.')

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM: 8.0 GB


In [7]:
# Load the sentence-transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded: all-MiniLM-L6-v2')
print(f'Embedding dimension: {model.get_sentence_embedding_dimension()}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\sharm\anaconda3\envs\recsys\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sharm\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: all-MiniLM-L6-v2
Embedding dimension: 384


---
## 5. Generate Embeddings

In [8]:
texts = df['embedding_text'].tolist()

# Use smaller batch size on CPU to avoid memory issues
batch_size = 256 if device == 'cuda' else 64

print(f'Encoding {len(texts):,} questions...')
print(f'Batch size: {batch_size} | Device: {device}')
if device == 'cuda':
    print(f'Estimated time: ~15-20 min')
else:
    print(f'Estimated time: ~2-3 hours (CPU)')
print()

start = time.time()

embeddings = model.encode(
    texts,
    batch_size=batch_size,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=device
)

elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s ({elapsed/60:.1f} min)')
print(f'Embeddings shape: {embeddings.shape}')
print(f'Memory: {embeddings.nbytes / 1024**2:.1f} MB')

Encoding 498,644 questions...
Batch size: 256 | Device: cuda
Estimated time: ~15-20 min



Batches:   0%|          | 0/1948 [00:00<?, ?it/s]


Done in 1080.9s (18.0 min)
Embeddings shape: (498644, 384)
Memory: 730.4 MB


---
## 6. Save Embeddings + ID Mapping

In [10]:
SAVE_DIR = '../Dataset_Cleaned'  # Save alongside cleaned data

# Save embeddings
np.save(os.path.join(SAVE_DIR, 'question_embeddings.npy'), embeddings)
emb_size = os.path.getsize(os.path.join(SAVE_DIR, 'question_embeddings.npy')) / (1024**2)
print(f'Saved: question_embeddings.npy ({emb_size:.1f} MB)')

# Save ID mapping — critical for correct FAISS lookups
question_ids = df['id'].values
np.save(os.path.join(SAVE_DIR, 'question_ids.npy'), question_ids)
print(f'Saved: question_ids.npy ({len(question_ids):,} IDs)')

# Verify alignment
print(f'\nAlignment check:')
print(f'  embeddings[0] → question_ids[0] = {question_ids[0]}')
print(f'  df.iloc[0]["id"] = {df.iloc[0]["id"]}')
print(f'  Match: {question_ids[0] == df.iloc[0]["id"]}')

Saved: question_embeddings.npy (730.4 MB)
Saved: question_ids.npy (498,644 IDs)

Alignment check:
  embeddings[0] → question_ids[0] = 11227809
  df.iloc[0]["id"] = 11227809
  Match: True


---
## 7. Sanity Check — Test Semantic Search

In [11]:
# Test 1: Python sorting question
test_query = 'how to sort a list in python'
query_vec = model.encode([test_query])

# Calculate cosine similarity with all embeddings
similarities = np.dot(embeddings, query_vec.T).flatten()
top_5 = similarities.argsort()[-5:][::-1]

print(f'Query: "{test_query}"')
print(f'Top 5 most similar questions:')
print('-' * 70)
for rank, idx in enumerate(top_5, 1):
    print(f'  {rank}. (similarity: {similarities[idx]:.3f}) {df.iloc[idx]["title"]}')
    print(f'     Tags: {df.iloc[idx]["tags"]}')
    print()

Query: "how to sort a list in python"
Top 5 most similar questions:
----------------------------------------------------------------------
  1. (similarity: 0.855) How to sort python list of strings of numbers
     Tags: python

  2. (similarity: 0.854) Sorting list of lists by the first element of each sub-list
     Tags: python

  3. (similarity: 0.850) Python list sort in descending order
     Tags: python,sorting,reverse

  4. (similarity: 0.846) Sort python list by function
     Tags: python,list,sorting

  5. (similarity: 0.845) Sort a list of lists by length and value in Python
     Tags: python,python-2.7,list



In [12]:
# Test 2: JavaScript async question (different domain)
test_query_2 = 'handle async await errors in javascript'
query_vec_2 = model.encode([test_query_2])

similarities_2 = np.dot(embeddings, query_vec_2.T).flatten()
top_5_2 = similarities_2.argsort()[-5:][::-1]

print(f'Query: "{test_query_2}"')
print(f'Top 5 most similar questions:')
print('-' * 70)
for rank, idx in enumerate(top_5_2, 1):
    print(f'  {rank}. (similarity: {similarities_2[idx]:.3f}) {df.iloc[idx]["title"]}')
    print(f'     Tags: {df.iloc[idx]["tags"]}')
    print()

Query: "handle async await errors in javascript"
Top 5 most similar questions:
----------------------------------------------------------------------
  1. (similarity: 0.787) async/await catch rejected Promises
     Tags: javascript,node.js,babeljs

  2. (similarity: 0.771) Can I throw error in an async function?
     Tags: javascript,promise,async-await,es6-promise

  3. (similarity: 0.768) Catching errors from nested async/await functions
     Tags: javascript,node.js,async-await,ecmascript-2017

  4. (similarity: 0.764) How to warn when you forget to `await` an async function in Javascript?
     Tags: javascript,webpack,babeljs

  5. (similarity: 0.764) Is there a way to wrap an await/async try/catch block to every function?
     Tags: javascript,express,async-await,ecmascript-2017



In [13]:
# Test 3: SQL question (another domain — should NOT match Python/JS)
test_query_3 = 'how to join two tables in SQL'
query_vec_3 = model.encode([test_query_3])

similarities_3 = np.dot(embeddings, query_vec_3.T).flatten()
top_5_3 = similarities_3.argsort()[-5:][::-1]

print(f'Query: "{test_query_3}"')
print(f'Top 5 most similar questions:')
print('-' * 70)
for rank, idx in enumerate(top_5_3, 1):
    print(f'  {rank}. (similarity: {similarities_3[idx]:.3f}) {df.iloc[idx]["title"]}')
    print(f'     Tags: {df.iloc[idx]["tags"]}')
    print()

Query: "how to join two tables in SQL"
Top 5 most similar questions:
----------------------------------------------------------------------
  1. (similarity: 0.808) how to join more than 2 table using sql Query?
     Tags: sql

  2. (similarity: 0.769) Join two tables with multiple foreign keys
     Tags: sql

  3. (similarity: 0.754) Multiple column left join from table1, table2
     Tags: mysql

  4. (similarity: 0.749) sql join two table
     Tags: mysql,sql,tsql,left-join

  5. (similarity: 0.740) sql join 2 rows in the same table
     Tags: mysql,sql,oracle



In [14]:
print('=' * 60)
print('EMBEDDING GENERATION COMPLETE')
print('=' * 60)
print(f'\n  Questions encoded:  {len(embeddings):,}')
print(f'  Embedding dim:      {embeddings.shape[1]}')
print(f'  Model:              all-MiniLM-L6-v2')
print(f'  Input format:       title [SEP] tags [SEP] body (200 words)')
print(f'\n  Files saved in {SAVE_DIR}/:')
print(f'    question_embeddings.npy  ({emb_size:.1f} MB)')
print(f'    question_ids.npy         ({len(question_ids):,} IDs)')
print(f'\n  Next step: Run faiss_index_search.ipynb')
print('=' * 60)

EMBEDDING GENERATION COMPLETE

  Questions encoded:  498,644
  Embedding dim:      384
  Model:              all-MiniLM-L6-v2
  Input format:       title [SEP] tags [SEP] body (200 words)

  Files saved in ../Dataset_Cleaned/:
    question_embeddings.npy  (730.4 MB)
    question_ids.npy         (498,644 IDs)

  Next step: Run faiss_index_search.ipynb
